# Sales vs. Flips Analysis (Entity-Level Audit)
**Goal:** Identify the true percentage of 'New Sales' that are actually 'Flips' vs 'Expansions' vs 'New Logos'.

### Classification Logic (Strictly following C1 Final Table):
1. **True New Logo:** The `ORG_URI` (Business Entity) has its first-ever activity in WEX history in the same month as the sale.
2. **Flip (C1 Definition):** The Entity opens a new account and matches the `HISTORICAL_ATTRITION_DECISION = 'Event: Flip/Conversion'` in your `C1` table.
3. **Organic Expansion:** The Entity already existed in WEX, but the new account was opened *without* triggering the flip/conversion event in `C1`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

### 1. Unified Audit Query
This query joins Salesforce data with your **Stage Table C1** to categorize every sale correctly.

In [ ]:
# analysis_sql = """
# WITH -- 1. Gross Sales (Approved accounts)
# SALES_UNIVERSE AS (
#     SELECT
#         WEX_ACCOUNT_NBR_C AS ACCOUNT_ID,
#         DATE_TRUNC('month', APP_APPROVED_DATE_C) AS SALE_MONTH,
#         APP_APPROVED_DATE_C AS SALE_DATE,
#         EXISTING_EXPOSURE_CHECK_C -- Añadimos el campo sugerido por Brad
#     FROM PREP.SALESFORCE.APPLICATION_REQUEST_C
#     WHERE WEX_ACCOUNT_NBR_C LIKE '91%'
#     AND LENGTH(WEX_ACCOUNT_NBR_C) = 13
#     AND APPLICATION_STATUS_C = 'Approved'
#     AND APP_APPROVED_DATE_C >= '2023-01-01'
# ),

# -- 2. Map Sales to Entities using Static Bridge
# SALES_MAPPED AS (
#     SELECT
#         s.*,
#         ao.ORG_URI
#     FROM SALES_UNIVERSE s
#     LEFT JOIN PREP.MDM_RELTIO.ENTITY_WXACCOUNTNUMBER wx ON s.ACCOUNT_ID = wx.ACCOUNTNUMBER
#     LEFT JOIN PREP.MDM_RELTIO.ENTITY_WXACCOUNTNUMBER_ORGANIZATION ao ON wx.URI = ao.URI
# ),

# -- 3. Audit against C1 Master Target Table
# ENTITY_AUDIT AS (
#     SELECT
#         sm.*,
#         (SELECT MAX(HISTORICAL_ATTRITION_DECISION) 
#          FROM WORKSPACE.digitalda_stage.entity_Christian_target_variable 
#          WHERE ORG_URI = sm.ORG_URI 
#            AND HISTORICAL_ATTRITION_DECISION = 'Event: Flip/Conversion' 
#            AND EVALUATION_MONTH BETWEEN DATEADD('month', -1, sm.SALE_MONTH) AND DATEADD('month', 1, sm.SALE_MONTH)) AS FLIP_DECISION,
#         (SELECT MIN(EVALUATION_MONTH) 
#          FROM WORKSPACE.digitalda_stage.entity_Christian_target_variable 
#          WHERE ORG_URI = sm.ORG_URI) AS ENTITY_BIRTH_DATE
#     FROM SALES_MAPPED sm
# ),

# -- 4. Final Classification
# SELECT
#     SALE_MONTH,
#     CASE
#         WHEN FLIP_DECISION = 'Event: Flip/Conversion' THEN 'Flip: Behavioral (C1)'
#         WHEN ORG_URI IS NULL THEN 'Unmapped to MDM (Cannot Determine)'
        
#         -- APLICAMOS EL FEEDBACK:
#         -- Solo lo consideramos Expansión si la entidad es vieja Y NO pasaron el check de Salesforce de nueva entidad.
#         -- Usamos COALESCE por los ~190k registros que vienen nulos/vacíos en ese campo.
#         WHEN ENTITY_BIRTH_DATE < DATEADD('month', -2, SALE_MONTH) 
#              AND COALESCE(EXISTING_EXPOSURE_CHECK_C, 'Failed') != 'Passed' THEN 'Flip: Existing Entity Expansion'
             
#         ELSE 'True New Logo'
#     END AS SALE_TYPE,
#     COUNT(*) AS SALES_COUNT
# FROM ENTITY_AUDIT
# GROUP BY 1, 2
# ORDER BY 1 DESC, 2;

# """


V2

WITH -- 1. Gross Sales (Approved accounts) + Channel & Size
SALES_UNIVERSE AS (
    SELECT
        a.WEX_ACCOUNT_NBR_C AS ACCOUNT_ID,
        DATE_TRUNC('month', a.APP_APPROVED_DATE_C) AS SALE_MONTH,
        a.APP_APPROVED_DATE_C AS SALE_DATE,
        a.EXISTING_EXPOSURE_CHECK_C,
        -- 1. Traemos el tamaño del deal en $ (Límite de crédito como proxy de Revenue)
        TRY_TO_NUMBER(a.APPROVED_CREDIT_LIMIT__C::VARCHAR) AS APPROVED_CREDIT_LIMIT,
        -- 2. Traemos el Canal de Venta (FS, IS, D2B)
        b.sales_type AS SALES_CHANNEL
    FROM PREP.SALESFORCE.APPLICATION_REQUEST_C a
    LEFT JOIN (
        -- Hacemos el mismo join seguro que usaste en tu C2
        SELECT SOURCE_ACCOUNT_ID, MAX(sales_type) AS sales_type
        FROM WORKSPACE.SALESMKTG.ACQ3_ACCOUNT
        GROUP BY 1
    ) b ON a.WEX_ACCOUNT_NBR_C = b.SOURCE_ACCOUNT_ID
    WHERE a.WEX_ACCOUNT_NBR_C LIKE '91%'
    AND LENGTH(a.WEX_ACCOUNT_NBR_C) = 13
    AND a.APPLICATION_STATUS_C = 'Approved'
    AND a.APP_APPROVED_DATE_C >= '2023-01-01'
),

-- 2. Map Sales to Entities using Static Bridge
SALES_MAPPED AS (
    SELECT
        s.*,
        ao.ORG_URI
    FROM SALES_UNIVERSE s
    LEFT JOIN PREP.MDM_RELTIO.ENTITY_WXACCOUNTNUMBER wx ON s.ACCOUNT_ID = wx.ACCOUNTNUMBER
    LEFT JOIN PREP.MDM_RELTIO.ENTITY_WXACCOUNTNUMBER_ORGANIZATION ao ON wx.URI = ao.URI
),

-- 3. Audit against C1 Master Target Table
ENTITY_AUDIT AS (
    SELECT
        sm.*,
        (SELECT MAX(HISTORICAL_ATTRITION_DECISION) 
         FROM WORKSPACE.digitalda_stage.entity_Christian_target_variable 
         WHERE ORG_URI = sm.ORG_URI 
           AND HISTORICAL_ATTRITION_DECISION = 'Event: Flip/Conversion' 
           AND EVALUATION_MONTH BETWEEN DATEADD('month', -1, sm.SALE_MONTH) AND DATEADD('month', 1, sm.SALE_MONTH)) AS FLIP_DECISION,
        (SELECT MIN(EVALUATION_MONTH) 
         FROM WORKSPACE.digitalda_stage.entity_Christian_target_variable 
         WHERE ORG_URI = sm.ORG_URI) AS ENTITY_BIRTH_DATE
    FROM SALES_MAPPED sm
),

-- 4. Final Classification (Agrupado por Canal y sumando Dólares)
SELECT
    SALE_MONTH,
    COALESCE(SALES_CHANNEL, 'Unknown') AS SALES_CHANNEL,
    CASE
        WHEN FLIP_DECISION = 'Event: Flip/Conversion' THEN 'Flip: Behavioral (C1)'
        WHEN ORG_URI IS NULL THEN 'Unmapped to MDM (Cannot Determine)'
        
        -- Cambié el nombre a "Program Change" (Malo) vs "Organic Expansion" (Bueno) para alinear con el jefe.
        WHEN ENTITY_BIRTH_DATE < DATEADD('month', -2, SALE_MONTH) 
             AND COALESCE(EXISTING_EXPOSURE_CHECK_C, 'Failed') != 'Passed' THEN 'Flip: Program Change (Uncaught by C1)'
             
        ELSE 'True New Logo / Organic Expansion'
    END AS SALE_TYPE,
    
    -- Ahora vemos la verdad:
    COUNT(*) AS SALES_TRANSACTION_COUNT,
    SUM(COALESCE(APPROVED_CREDIT_LIMIT, 0)) AS TOTAL_CREDIT_LIMIT_USD

FROM ENTITY_AUDIT
GROUP BY 1, 2, 3
ORDER BY 1 DESC, 2, 3;


print("SQL Query finalized using ONLY columns available in the C1 final table schema.")


### 2. Analysis & Reporting
Run this code to visualize the impact on Sales metrics.

In [ ]:
# df = pd.read_sql(analysis_sql, conn)

def generate_viz(df):
    pivot_df = df.pivot(index='SALE_MONTH', columns='SALE_TYPE', values='SALES_COUNT').fillna(0)
    pivot_df['Total Sales'] = pivot_df.sum(axis=1)
    
    # Metrics for the boss
    pivot_df['Flip Rate %'] = pivot_df['Flip (C1: Confirmed Conversion)'] / pivot_df['Total Sales'] * 100
    pivot_df['Expansion Rate %'] = pivot_df['Expansion (Existing Entity)'] / pivot_df['Total Sales'] * 100
    
    fig, ax1 = plt.subplots(figsize=(14, 8))
    
    # Stacked Bars
    colors = ['#e67e22', '#3498db', '#27ae60']
    cols = ['Flip (C1: Confirmed Conversion)', 'Expansion (Existing Entity)', 'True New Logo']
    pivot_df[cols].plot(kind='bar', stacked=True, ax=ax1, color=colors, alpha=0.8)
    ax1.set_ylabel('Count of Sales')
    
    # Flip Rate Line
    ax2 = ax1.twinx()
    ax2.plot(pivot_df.index.strftime('%Y-%m'), pivot_df['Flip Rate %'], color='red', marker='o', linewidth=2, label='Flip Rate % (C1)')
    ax2.set_ylabel('Flip Rate %')
    ax2.set_ylim(0, 100)
    
    plt.title('Monthly Sales Audit: Distinguishing Flips from Organic Expansion')
    plt.legend(loc='upper left')
    plt.show()
    
    print(f"--- Audit Summary ---")
    print(f"Avg Flip Rate (C1): {pivot_df['Flip Rate %'].mean():.1f}%")
    print(f"Avg Expansion Rate: {pivot_df['Expansion Rate %'].mean():.1f}%")
    return pivot_df


### 3. Entity Accounts Audit
This query tracks the average number of accounts per entity over time to determine if a sudden drop (or increase) in accounts per entity is driving a spike in behavioral flips.

In [ ]:
entity_accounts_sql = """
SELECT 
    EVALUATION_MONTH,
    AVG(ACCOUNT_COUNT) AS AVG_TOTAL_ACCOUNTS,
    AVG(ACTIVE_COUNT) AS AVG_ACTIVE_ACCOUNTS,
    SUM(ACCOUNT_COUNT) AS TOTAL_SYSTEM_ACCOUNTS,
    SUM(ACTIVE_COUNT) AS TOTAL_ACTIVE_SYSTEM_ACCOUNTS,
    COUNT(DISTINCT ORG_URI) AS TOTAL_ENTITIES
FROM WORKSPACE.digitalda_stage.entity_Christian_target_variable
GROUP BY EVALUATION_MONTH
ORDER BY EVALUATION_MONTH DESC;
"""

print("Entity accounts audit query ready.")

In [ ]:
# df_accounts = pd.read_sql(entity_accounts_sql, conn)

def plot_entity_accounts(df):
    # Sort chronologically for plotting
    df = df.sort_values('EVALUATION_MONTH')
    
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    # Plot Total Entities
    ax1.plot(df['EVALUATION_MONTH'], df['TOTAL_ENTITIES'], color='#34495e', marker='s', linewidth=2, label='Total Entities')
    ax1.set_xlabel('Month')
    ax1.set_ylabel('Total Entities')
    ax1.grid(True, linestyle='--', alpha=0.7)
    
    # Plot Average Active Accounts per Entity on a secondary axis
    ax2 = ax1.twinx()
    ax2.plot(df['EVALUATION_MONTH'], df['AVG_ACTIVE_ACCOUNTS'], color='#e74c3c', marker='o', linewidth=2, label='Avg Active Accounts per Entity')
    ax2.set_ylabel('Avg Active Accounts / Entity')
    
    # Title and Legend
    plt.title('Entity Population & Average Active Accounts Over Time')
    # Workaround to combine legends from two axes
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    lines_2, labels_2 = ax2.get_legend_handles_labels()
    ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')
    
    plt.show()
    
    # Identify potential anomalies (e.g., month-over-month drop in avg accounts)
    df['MOM_Avg_Accounts_Change'] = df['AVG_ACTIVE_ACCOUNTS'].pct_change() * 100
    print("--- Recent Months ---")
    print(df[['EVALUATION_MONTH', 'AVG_ACTIVE_ACCOUNTS', 'MOM_Avg_Accounts_Change']].tail(12))
    
    return df